In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *

# Case Western sessions: checks, then compile

Case Western tested 7 patients (OFF and ON medication). Unlike Harvard and Stanford there are no block files: each
session is a single MAT file at the top of `raw/case_western_data/` (e.g. `Brady_LHO_OFFmeds_rl.mat`), with one
practice file in `practice/` (SGA OFF). These files are the rawest data there is, so they are what is compiled.

Without per-block files and filenames there is no three-clock check. What can be checked:

1. **Header time**: the MAT header's `Created on` (when the file was last saved);
2. **Clock**: the `time` matrix is one monotonic clock; within a session it must never run backwards (one program
   run, not files stitched from different PC sessions), and its gaps show the pauses;
3. **Design**: the colour splits and prior over the whole session and in consecutive 160-trial stretches (the size of
   a block at the other sites), which should all imply the same prior;
4. **OFF and ON**: which was recorded first and how far apart, where both sit on one clock;
5. **Practice**: recorded before its session; and **600+ trials**.

Writes `compiled/case_western/{blocks.csv, session_summary.csv, sessions/<session>.parquet}` with the same columns
as the other block sites, each session being a single block. Sessions are named by patient and medication state
only (`LHO_off`); the subtype prefix and prior label in the raw file names are not carried over.

In [2]:
import re

import scipy.io

from src.data_io import catalog
from src.utils import blocks as blk
from src.utils.sessions import decode_events

CASE_WESTERN = catalog.path("raw_case_western_sessions")
SESSION_FILE = re.compile(r"^(?:(?P<subtype>Brady|Trem)_)?(?P<subject>[A-Z]+)_(?P<med>ON|OFF)meds_?(?P<rest>.*)$")
PAUSE_S = 20  # a gap between trial starts longer than this is a pause
CHUNK = 160  # trials per stretch for the design-over-time check
BATCH_SAVE_MIN_FILES = 3  # this many files saved within BATCH_SAVE_S of each other were re-saved together, not recorded then
BATCH_SAVE_S = 5

## 1. Inventory

In [3]:
rows, decoded = [], {}
for path in sorted(CASE_WESTERN.rglob("*")):
    if path.is_dir():
        continue
    found = SESSION_FILE.match(path.stem)
    if path.suffix != ".mat" or not found:
        print(f"not a session file: {path.relative_to(CASE_WESTERN)}")
        continue
    phase = "practice" if "practice" in found["rest"].lower() or path.parent.name == "practice" else "trial"
    medication = found["med"].lower()
    session_id = f"{found['subject']}_{medication}"
    mat = scipy.io.loadmat(path)
    event, time = mat["event"], mat["time"]
    file = f"{session_id}/{phase}"  # the raw file name carries labels that are not saved
    trials = decode_events(event, time)
    decoded[file] = trials
    starts = time[:, 0] / 1000
    gaps = np.diff(starts)
    pauses = gaps[gaps > PAUSE_S]
    rows.append({
        "session_id": session_id, "subject_id": found["subject"], "group": "pd", "medication": medication, "phase": phase,
        "file": file, "path": path.relative_to(CASE_WESTERN).as_posix(), "header_time": blk.header_time(mat),
        "n_rows": event.shape[0], "n_cols": event.shape[1], "clock_start_s": starts[0], "clock_end_s": time[time > 0].max() / 1000,
        "clock_backwards": int((gaps < 0).sum()), "n_pauses": len(pauses), "longest_pause_min": pauses.max() / 60 if len(pauses) else 0.0,
        "valid_pct": 100 * trials.is_valid.mean(),
    })
blocks = pd.DataFrame(rows)
blocks["duration_min"] = (blocks.clock_end_s - blocks.clock_start_s) / 60
print(f"{len(blocks)} files: {(blocks.phase == 'trial').sum()} sessions of {blocks.subject_id.nunique()} patients, "
      f"{(blocks.phase == 'practice').sum()} practice file(s)")
blocks[["session_id", "phase", "header_time", "n_rows", "duration_min", "n_pauses", "longest_pause_min", "clock_backwards", "valid_pct"]].round(1)

15 files: 14 sessions of 7 patients, 1 practice file(s)


,session_id,phase,header_time,n_rows,duration_min,n_pauses,longest_pause_min,clock_backwards,valid_pct
0,LHO_off,trial,2022-11-14 10:14:23,2236,42.1,3,1.5,0,24.6
1,LHO_on,trial,2022-11-14 10:14:23,2713,44.5,2,3.9,0,22.4
2,RBA_off,trial,2022-11-14 10:14:23,1036,33.1,0,0.0,0,65.2
3,RBA_on,trial,2022-11-14 10:14:23,963,31.3,0,0.0,0,71.2
4,RDE_off,trial,2022-11-14 10:14:23,1497,37.0,5,1.5,0,43.7
5,RDE_on,trial,2022-11-14 10:14:23,1528,41.7,6,1.7,0,39.9
6,RSH_off,trial,2022-11-14 10:14:23,1705,32.3,2,2.2,0,40.1
7,RSH_on,trial,2022-11-14 10:14:23,1470,37.8,6,2.2,0,37.8
8,RZA_off,trial,2022-11-14 10:14:23,1885,54.0,6,16.7,0,31.8
9,RZA_on,trial,2022-11-14 10:14:23,2165,37.8,5,1.1,0,23.7


## 2. Header times

Files saved within a few seconds of each other were re-saved together, so their header is the date of that batch,
not of the session.

In [4]:
by_time = blocks.sort_values("header_time")
batch = (by_time.header_time.diff().dt.total_seconds() > BATCH_SAVE_S).cumsum()
blocks["header_is_batch_save"] = batch.map(batch.value_counts()).reindex(blocks.index) >= BATCH_SAVE_MIN_FILES
for _, group in blocks[blocks.header_is_batch_save].groupby(batch):
    print(f"{len(group)} files re-saved together at {group.header_time.min()}-{group.header_time.max():%H:%M:%S}: {', '.join(group.session_id)}")
print("original headers:", ", ".join(f"{r.session_id} {r.phase} {r.header_time}" for r in blocks[~blocks.header_is_batch_save].itertuples()))

12 files re-saved together at 2022-11-14 10:14:23-10:14:24: LHO_off, LHO_on, RBA_off, RBA_on, RDE_off, RDE_on, RSH_off, RSH_on, RZA_off, RZA_on, SNO_off, SNO_on
original headers: SGA_off trial 2018-08-17 08:58:01, SGA_on trial 2018-08-17 09:55:55, SGA_off practice 2018-08-17 08:23:05


## 3. Design over the session

The whole session's colour splits and implied prior (shared code with the other sites), then the prior implied by
each consecutive stretch of 160 colour-shown trials (the last stretch only if it has at least 80).

In [5]:
blk.add_design(blocks, decoded)


def stretch_priors(trials):
    shown = trials.dropna(subset=["color", "target"]).reset_index(drop=True)
    priors = []
    for start in range(0, len(shown), CHUNK):
        part = shown.iloc[start:start + CHUNK]
        if len(part) >= CHUNK // 2:
            right = part.target.eq("right").groupby(part.color).mean() * 100
            priors.append(blk.implied_prior(right.get("green", 50), right.get("red", 50)))
    return priors


blocks["stretch_priors"] = blocks.file.map(lambda f: " | ".join(stretch_priors(decoded[f])))
blocks["prior_consistent"] = blocks.stretch_priors.str.split(" \\| ").map(lambda p: len(set(p)) == 1)
blocks[["session_id", "phase", "n_shown", "coherences", "green_right_pct", "red_right_pct", "block_prior", "design_error", "stretch_priors"]].round(1)

,session_id,phase,n_shown,coherences,green_right_pct,red_right_pct,block_prior,design_error,stretch_priors
0,LHO_off,trial,635,"0,13,35,100",50.3,25.2,rl,0.0,rl | rl | rl | rl
1,LHO_on,trial,650,"0,13,35,100",25.5,50.3,gl,0.0,gl | gl | gl | gl
2,RBA_off,trial,700,"0,13,35,100",75.1,50.3,gr,0.0,gr | gr | gr | gr
3,RBA_on,trial,700,"0,13,35,100",50.1,24.8,rl,0.0,rl | rl | rl | rl
4,RDE_off,trial,700,"0,13,35,100",50.0,74.9,rr,0.0,rr | rr | rr | rr
5,RDE_on,trial,702,"0,13,35,100",24.8,50.1,gl,0.0,gl | gl | gl | gl
6,RSH_off,trial,701,"0,13,35,100",50.7,25.0,rl,0.0,rl | rl | rl | rl
7,RSH_on,trial,700,"0,13,35,100",24.6,50.1,gl,0.0,gl | gl | gl | gl
8,RZA_off,trial,669,"0,13,35,100",24.6,50.7,gl,0.0,gl | gl | gl | gl
9,RZA_on,trial,638,"0,13,35,100",49.8,24.8,rl,0.0,rl | rl | rl | rl


## 4. OFF and ON on the clock

Where a patient's OFF and ON sessions sit on one continuous clock (the PC was not restarted between them), the clock
gives their order and the awake time between them. The clock pauses while the PC sleeps, so a short clock gap does not
mean a short wall-clock gap. If the PC was restarted in between, the clock order says nothing; a patient whose second
session starts at a lower clock value than the first ended cannot be ordered.

In [6]:
sessions = blocks[blocks.phase == "trial"].set_index("session_id")
pairs = []
for subject, group in sessions.groupby("subject_id", sort=False):
    if set(group.medication) != {"off", "on"}:
        pairs.append({"subject_id": subject, "order_by_clock": f"only {', '.join(group.medication)}"})
        continue
    off, on = group[group.medication == "off"].iloc[0], group[group.medication == "on"].iloc[0]
    first, second = (off, on) if off.clock_start_s < on.clock_start_s else (on, off)
    gap = (second.clock_start_s - first.clock_end_s) / 60
    pairs.append({"subject_id": subject, "order_by_clock": f"{first.medication.upper()} then {second.medication.upper()}",
                  "awake_gap_min": gap, "header_gap_min": (second.header_time - first.header_time).total_seconds() / 60
                  if not (first.header_is_batch_save or second.header_is_batch_save) else np.nan})
pairs = pd.DataFrame(pairs)
pairs["same_sitting"] = pairs.awake_gap_min.between(0, 60)
pairs.round(1)

,subject_id,order_by_clock,awake_gap_min,header_gap_min,same_sitting
0,LHO,OFF then ON,6.7,NaN,True
1,RBA,OFF then ON,13.6,NaN,True
2,RDE,OFF then ON,19.6,NaN,True
3,RSH,ON then OFF,4064.0,NaN,False
4,RZA,OFF then ON,10.8,NaN,True
5,SGA,OFF then ON,3.2,57.9,True
6,SNO,OFF then ON,11173.7,NaN,False


## 5. Practice

In [7]:
practice = blocks[blocks.phase == "practice"]
for p in practice.itertuples():
    session = sessions.loc[p.session_id]
    before = p.clock_end_s <= session.clock_start_s and (p.header_is_batch_save or session.header_is_batch_save or p.header_time <= session.header_time)
    print(f"{p.session_id}: practice {p.n_shown:.0f} trials at coherences {p.coherences}, prior {p.block_prior}; "
          f"ends {(session.clock_start_s - p.clock_end_s) / 60:.1f} min (clock) before the session starts: "
          f"{'before' if before else 'NOT before'} the session")

SGA_off: practice 124 trials at coherences 100, prior none; ends 2.6 min (clock) before the session starts: before the session


## 6. Session summary

In [8]:
blocks["use"] = blocks.n_cols == 10
summary = sessions.reset_index()[["session_id", "subject_id", "group", "medication", "header_time", "header_is_batch_save", "duration_min",
                                  "n_pauses", "longest_pause_min", "clock_backwards", "n_rows", "n_shown", "valid_pct", "block_prior",
                                  "stretch_priors", "prior_consistent", "design_error"]]
summary = summary.rename(columns={"n_rows": "n_trials", "block_prior": "session_prior", "design_error": "max_design_error"})
summary["design_ok"] = summary.max_design_error <= blk.DESIGN_TOLERANCE
summary["n_trial_blocks"] = 1
summary["n_practice_blocks"] = summary.session_id.map(practice.groupby("session_id").size()).fillna(0).astype(int)
summary["reaches_600"] = summary.n_shown >= blk.MIN_SESSION_TRIALS
summary = summary.merge(pairs[["subject_id", "order_by_clock", "awake_gap_min", "same_sitting"]], on="subject_id", how="left")


def issues(s):
    found = []
    if s.clock_backwards:
        found.append(f"clock runs backwards {s.clock_backwards} time(s)")
    if not s.prior_consistent:
        found.append(f"prior changes ({s.stretch_priors})")
    if not s.design_ok:
        found.append(f"design off 75:25 / 50:50 (error {s.max_design_error:.2f})")
    if not s.n_practice_blocks:
        found.append("no practice file")
    if not s.reaches_600:
        found.append(f"{s.n_shown:.0f} trials")
    return "; ".join(found)


summary["issues"] = summary.apply(issues, axis=1)
summary[["session_id", "duration_min", "n_pauses", "n_shown", "valid_pct", "session_prior", "prior_consistent", "design_ok",
         "order_by_clock", "awake_gap_min", "issues"]].round(1)

,session_id,duration_min,n_pauses,n_shown,valid_pct,session_prior,prior_consistent,design_ok,order_by_clock,awake_gap_min,issues
0,LHO_off,42.1,3,635,24.6,rl,True,True,OFF then ON,6.7,no practice file
1,LHO_on,44.5,2,650,22.4,gl,True,True,OFF then ON,6.7,no practice file
2,RBA_off,33.1,0,700,65.2,gr,True,True,OFF then ON,13.6,no practice file
3,RBA_on,31.3,0,700,71.2,rl,True,True,OFF then ON,13.6,no practice file
4,RDE_off,37.0,5,700,43.7,rr,True,True,OFF then ON,19.6,no practice file
5,RDE_on,41.7,6,702,39.9,gl,True,True,OFF then ON,19.6,no practice file
6,RSH_off,32.3,2,701,40.1,rl,True,True,ON then OFF,4064.0,no practice file
7,RSH_on,37.8,6,700,37.8,gl,True,True,ON then OFF,4064.0,no practice file
8,RZA_off,54.0,6,669,31.8,gl,True,True,OFF then ON,10.8,no practice file
9,RZA_on,37.8,5,638,23.7,rl,True,True,OFF then ON,10.8,no practice file


## 7. Findings

Written out from the tables.

In [9]:
s = summary.set_index("session_id")
lines = [
    f"{len(s)} sessions of {s.subject_id.nunique()} patients, one file each; practice file for {', '.join(practice.session_id) or 'none'}.",
    f"Header times re-saved in a batch: {', '.join(s.index[s.header_is_batch_save])}; original: {', '.join(s.index[~s.header_is_batch_save]) or 'none'}.",
    f"Clock runs backwards within a session: {', '.join(s.index[s.clock_backwards > 0]) or 'none'} (every session is one continuous run).",
    f"Pauses over {PAUSE_S} s: median {s.n_pauses.median():.0f} per session, longest {s.longest_pause_min.max():.1f} min ({s.longest_pause_min.idxmax()}).",
    f"Prior changes within a session: {', '.join(s.index[~s.prior_consistent]) or 'none'}; design off 75:25 / 50:50: {', '.join(s.index[~s.design_ok]) or 'none'}.",
    f"Trials: {s.n_shown.min():.0f}-{s.n_shown.max():.0f} (median {s.n_shown.median():.0f}); under {blk.MIN_SESSION_TRIALS}: {', '.join(s.index[~s.reaches_600]) or 'none'}; "
    f"under 700: {', '.join(f'{i} ({n:.0f})' for i, n in s.n_shown[s.n_shown < 700].items()) or 'none'}.",
    f"Valid trials: {s.valid_pct.min():.0f}-{s.valid_pct.max():.0f}%; under 35%: {', '.join(f'{i} ({v:.0f}%)' for i, v in s.valid_pct[s.valid_pct < 35].items()) or 'none'}.",
    "OFF and ON on one clock: " + "; ".join(f"{r.subject_id} {r.order_by_clock}, {r.awake_gap_min:.0f} min awake time apart"
                                            for r in pairs[pairs.same_sitting].itertuples()) + ".",
    "OFF and ON not in one sitting on the clock: " + ("; ".join(f"{r.subject_id} ({r.order_by_clock} if the PC was not restarted in between, {r.awake_gap_min / 60:.0f} h awake time apart)"
                                                             for r in pairs[~pairs.same_sitting].itertuples()) or "none") + ".",
    f"Original headers: " + ", ".join(f"{r.session_id} {r.phase} saved {r.header_time}" for r in blocks[~blocks.header_is_batch_save].itertuples()) + ".",
]
print("\n".join(f"- {line}" for line in lines))

- 14 sessions of 7 patients, one file each; practice file for SGA_off.
- Header times re-saved in a batch: LHO_off, LHO_on, RBA_off, RBA_on, RDE_off, RDE_on, RSH_off, RSH_on, RZA_off, RZA_on, SNO_off, SNO_on; original: SGA_off, SGA_on.
- Clock runs backwards within a session: none (every session is one continuous run).
- Pauses over 20 s: median 2 per session, longest 16.7 min (RZA_off).
- Prior changes within a session: none; design off 75:25 / 50:50: none.
- Trials: 635-706 (median 700); under 600: none; under 700: LHO_off (635), LHO_on (650), RZA_off (669), RZA_on (638), SNO_on (650).
- Valid trials: 22-71%; under 35%: LHO_off (25%), LHO_on (22%), RZA_off (32%), RZA_on (24%).
- OFF and ON on one clock: LHO OFF then ON, 7 min awake time apart; RBA OFF then ON, 14 min awake time apart; RDE OFF then ON, 20 min awake time apart; RZA OFF then ON, 11 min awake time apart; SGA OFF then ON, 3 min awake time apart.
- OFF and ON not in one sitting on the clock: RSH (ON then OFF if the PC was 

## 8. Compile

Each session (with its practice file, if any) as one trial table with the shared session and block columns. Each
session is a single block, so `block` is 1 and `block_file` is `<session>/trial` (or `/practice`).

In [10]:
blocks["filename_time"] = pd.NaT  # one block per phase, so there is nothing to order
blk.compile_sessions(blocks, decoded, summary, "compiled_case_western_session")
catalog.save(blocks.drop(columns=["path", "filename_time"]), "compiled_case_western_blocks")
catalog.save(summary, "compiled_case_western_session_summary")

saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/LHO_off.parquet


saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/LHO_on.parquet


saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/RBA_off.parquet
saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/RBA_on.parquet
saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/RDE_off.parquet


saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/RDE_on.parquet


saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/RSH_off.parquet


...

saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/RZA_off.parquet


saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/RZA_on.parquet


saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/SGA_off.parquet
saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/SGA_on.parquet


saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/SNO_off.parquet
saved compiled_case_western_session -> /mnt/pd-data/compiled/case_western/sessions/SNO_on.parquet
saved compiled_case_western_blocks -> /mnt/pd-data/compiled/case_western/blocks.csv


saved compiled_case_western_session_summary -> /mnt/pd-data/compiled/case_western/session_summary.csv
